# Criando um conjunto de testes
Na teoria, criar um conjunto de testes é simples: escolher algumas instâncias aleatórias, geralmente 20% do conjunto de dados e defini-las

In [3]:
import numpy as np
import pandas as pd

In [11]:
def shuffle_and_split_data(data, test_ratio):
    shuffle_indices = np.random.permutation(len(data))
    test_set_size = int(len(data)*test_ratio)
    test_indices = shuffle_indices[:test_set_size]
    train_indices = shuffle_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

## Loading Data

In [9]:
housing = pd.read_csv("datasets/housing/housing.csv")
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Criando conjuntos de Treino e Teste

In [12]:
train_set, test_set = shuffle_and_split_data(housing, 0.2)

In [17]:
display(train_set.head())
print(len(train_set))

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
10179,-118.21,33.93,30.0,2831.0,862.0,3649.0,883.0,1.9668,152100.0,<1H OCEAN
4188,-120.70,38.69,13.0,4492.0,821.0,2093.0,734.0,4.0709,151700.0,INLAND
16697,-122.25,38.00,16.0,2978.0,411.0,1531.0,400.0,6.5006,237700.0,NEAR BAY
7624,-118.27,34.07,42.0,1175.0,428.0,1593.0,407.0,2.3438,213300.0,<1H OCEAN
9683,-117.21,32.82,31.0,2035.0,383.0,866.0,360.0,3.8529,212000.0,NEAR OCEAN


16512


In [18]:
display(test_set.head())
print(len(test_set))

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
9641,-118.29,33.98,46.0,1118.0,300.0,786.0,254.0,1.4042,110000.0,<1H OCEAN
12322,-118.26,34.15,6.0,3340.0,945.0,2315.0,846.0,2.8884,252300.0,<1H OCEAN
7152,-120.61,35.13,16.0,3431.0,721.0,1777.0,701.0,2.7301,190400.0,<1H OCEAN
6061,-122.66,38.47,23.0,2246.0,437.0,1035.0,386.0,3.7617,172600.0,<1H OCEAN
16286,-118.20,33.90,33.0,1435.0,322.0,1298.0,299.0,2.7813,105100.0,<1H OCEAN


4128


Isso funciona mas, ao executar o programa novamente, um novo conjunto de testes diferente vai ser gerado e assim, ao longo do tempo o seu algoritmo acabaria vendo todo o conjunto de dados.

Uma solução aparente seria salvar o conjunto de testes na primeira execução e depois só carregar isso em execuções posteriores. Ou então, 
gerar números aleatórios com ``random.seed()`` andes de chamar ``random.permutation()`` e garantir que ele sempre gere os mesmos índices embaralhados.

Entretanto, essas duas abordagens quebram quando você altera as instâncias do conjunto de dados.

<br>

- Se você salva o conjunto de testes na primeira execução:
    
    <table>
        <tr>
            <td>
            <strong>1ª execução (100 instâncias):</strong><br>
            Treino: [0, 1, 2, ... 79]<br>
            Teste: [80, 81, ... 99] <em>← salvo em disco</em><br>
            </td>
        </tr>
    </table> 
        
    Agora o dataset é atualizado para 120 instâncias. Você carrega o teste salvo — as mesmas instâncias de antes. OK, sem contaminação do conjunto de treino.

    - O problema: As 20 novas instâncias (100 a 119) ficam todas no treino automaticamente.

    - Com o tempo, o modelo nunca vai ser testado em dados novos. Assim:
    <p align="center"><i>O teste fica "envelhecido" e pode deixar de ser representativo.</i></p>

    - Além disso, se o dataset for completamente substituído (não apenas expandido), os índices salvos podem não fazer mais sentido nenhum.

<br>

- Se você aplica uma ``seed``:

    - A seed garante que o embaralhamento seja sempre igual para o mesmo conjunto de dados. Mas se o dataset mudar (novas linhas adicionadas), o embaralhamento produz índices diferentes, porque agora existem mais elementos. Portanto:
        <p align="center"><i>Instâncias que estavam no treino podem acabar no teste, e vice-versa.</i></p>

    - Pense assim:
        <p align="center"><i>O teste existe para simular o mundo real, dados que o modelo nunca viu. Se uma instância estava no treino antes, e depois de um update ela migra pro teste, você está avaliando o modelo com dados que ele já conhece. O resultado parece bom, mas é uma ilusão.
        </i></p>
    
    - A sequência do problema:

        <table>
            <tr>
                <td>
                <strong>1ª execução (100 instâncias):</strong><br>
                Treino: instâncias [0 a 79]<br>
                Teste:  instâncias [80 a 99]<br>
                <br>
                <strong>Dataset atualizado (120 instâncias) + seed fixa:</strong><br>
                Treino: instâncias [3, 7, 15, 82...] ← embaralhamento novo <br>
                Teste:  instâncias [91, 4, 67...]  ← instância 4 estava no treino antes!<br>
                </td>
            </tr>
        </table> 